In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timezone

# ------------------------------------------------------------
# STEP 1: CONFIGURATION
# ------------------------------------------------------------

BASE_PATH = "/Volumes/smart_city/pollution/raw_data"

CATALOG = "smart_city"
SCHEMA = "pollution"

print("Starting Bronze Layer...")

Starting Bronze Layer...


In [0]:
# ------------------------------------------------------------
# STEP 2: READ RAW AIR QUALITY DATA
# ------------------------------------------------------------

air_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{BASE_PATH}/air_quality/")
)

print("Air Quality raw records:", air_raw.count())

# ------------------------------------------------------------
# STEP 3: READ RAW WEATHER DATA
# ------------------------------------------------------------

weather_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{BASE_PATH}/weather/")
)

print("Weather raw records:", weather_raw.count())

Air Quality raw records: 2016
Weather raw records: 2016


In [0]:
# ------------------------------------------------------------
# STEP 4: READ RAW NASA POWER DATA
# ------------------------------------------------------------

nasa_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{BASE_PATH}/nasa_power/")
)

print("NASA POWER raw records:", nasa_raw.count())

# ------------------------------------------------------------
# STEP 5: ADD BRONZE METADATA
# ------------------------------------------------------------

def add_bronze_metadata(df, source_name):

    return (
        df
        .withColumn(
            "ingestion_timestamp",
            F.current_timestamp()
        )
        .withColumn(
            "source_system",
            F.lit(source_name)
        )
    )

air_bronze = add_bronze_metadata(
    air_raw,
    "Open-Meteo Air Quality"
)

weather_bronze = add_bronze_metadata(
    weather_raw,
    "Open-Meteo Weather"
)

nasa_bronze = add_bronze_metadata(
    nasa_raw,
    "NASA POWER"
)

NASA POWER raw records: 84


In [0]:
# ------------------------------------------------------------
# STEP 6: CREATE BRONZE DELTA TABLES
# ------------------------------------------------------------

def save_bronze_table(df, table_name):

    full_table_name = f"{CATALOG}.{SCHEMA}.{table_name}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_table_name)
    )

    print(f"{full_table_name} created successfully!")

save_bronze_table(
    air_bronze,
    "bronze_air_quality"
)

save_bronze_table(
    weather_bronze,
    "bronze_weather"
)

save_bronze_table(
    nasa_bronze,
    "bronze_nasa_power"
)

smart_city.pollution.bronze_air_quality created successfully!
smart_city.pollution.bronze_weather created successfully!
smart_city.pollution.bronze_nasa_power created successfully!


In [0]:
# ------------------------------------------------------------
# STEP 7: VALIDATE BRONZE TABLES
# ------------------------------------------------------------

print("\n========== BRONZE VALIDATION ==========")

tables = [
    "bronze_air_quality",
    "bronze_weather",
    "bronze_nasa_power"
]

for table in tables:

    df = spark.table(f"{CATALOG}.{SCHEMA}.{table}")

    print(f"\nTable: {table}")
    print("Total Records:", df.count())

    print("Columns:", df.columns)

    print("Schema:")
    df.printSchema()

# ------------------------------------------------------------
# STEP 8: FINAL STATUS
# ------------------------------------------------------------

print("\n===================================")
print("BRONZE LAYER EXECUTION COMPLETED")
print("===================================")

display(
    spark.sql(
        f"SHOW TABLES IN {CATALOG}.{SCHEMA}"
    )
)



========== BRONZE VALIDATION ==========

Table: bronze_air_quality
Total Records: 2016
Columns: ['city', 'timestamp', 'pm10', 'pm2_5', 'nitrogen_dioxide', 'sulphur_dioxide', 'carbon_monoxide', 'ozone', 'source', 'date', 'extracted_at_utc', 'ingestion_timestamp', 'source_system']
Schema:
root
 |-- city: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- pm10: string (nullable = true)
 |-- pm2_5: string (nullable = true)
 |-- nitrogen_dioxide: string (nullable = true)
 |-- sulphur_dioxide: string (nullable = true)
 |-- carbon_monoxide: string (nullable = true)
 |-- ozone: string (nullable = true)
 |-- source: string (nullable = true)
 |-- date: string (nullable = true)
 |-- extracted_at_utc: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_system: string (nullable = true)


Table: bronze_weather
Total Records: 2016
Columns: ['city', 'timestamp', 'temperature', 'humidity', 'wind_speed', 'precipitation', 'source', 'date', 'e

database,tableName,isTemporary
pollution,bronze_air_quality,false
pollution,bronze_nasa_power,false
pollution,bronze_weather,false
pollution,etl_gold_city_daily,false
pollution,etl_gold_pollutant_trends,false
pollution,etl_gold_pollution_hotspots,false
pollution,etl_silver_air_quality,false
pollution,etl_silver_nasa_power,false
pollution,etl_silver_weather,false
pollution,gold_city_daily,false
